# Per-second counting uncertainty

A size distribution measured over one second rests on a small number of particles in each bin, so
each value carries a counting (Poisson) uncertainty of one over the square root of the count. The
enhancements package computes it alongside the product without changing any stored number:
`per_bin_relative_uncertainty`, `per_bin_absolute_uncertainty`, `is_statistically_significant`
and `propagate_to_total` in `arrayprobe_enhancements.poisson_uncertainty`. This notebook applies
them to a seeded random (time, bin) count array shaped like a one-second product, and checks the
estimate against the scatter of repeated draws. It runs in seconds and needs no data files.

**Before this:** [06_read_products](../1_getting_started/06_read_products.ipynb) (Reading the output files). **Related:** [03_explicit_bin_edges](03_explicit_bin_edges.ipynb). **Needs:** the enhancements package.

This notebook reads files from the arrayprobe source repository, so run it from a clone of that repository; `pip install arrayprobe` alone does not include them.

In [1]:
# Launch guard: the cells below reach scripts/ and tests/data/ relative to the repository root.
from pathlib import Path

if not (Path("scripts").is_dir() and Path("tests/data").is_dir()):
    raise RuntimeError(
        f"This notebook must run with the repository root as its working directory, not {Path.cwd()}. "
        "Jupyter starts a kernel in the notebook's own folder (examples/6_science/), so run "
        "`%cd ../..` in a new cell, then run this cell again. "
        "Headless, from the repository root: python scripts/execute_example_notebooks.py --check <name>"
    )

## A demo one-second product

Sixty seconds by twenty bins of Poisson counts with a falling mean per bin, a concentration made by
dividing by a fixed sampled volume, and one second with no coverage (all `NaN`).

In [2]:
import numpy as np
from arrayprobe_enhancements import poisson_uncertainty as pu

rng = np.random.default_rng(43)
n_sec, n_bin = 60, 20
mean_per_bin = 400.0 * np.exp(-0.35 * np.arange(n_bin))   # counts per second per bin
count = rng.poisson(mean_per_bin, size=(n_sec, n_bin)).astype(float)
count[17, :] = np.nan                                     # a second with no coverage
volume_l = 2.5                                            # sampled volume per second, litres
conc = count / volume_l                                   # per litre
print("counts in bin 0, 5, 10, 19 (second 0):", count[0, [0, 5, 10, 19]])
assert count.shape == (n_sec, n_bin)
assert np.isnan(count[17]).all()

counts in bin 0, 5, 10, 19 (second 0): [409.  68.  10.   0.]


## Relative and absolute uncertainty per bin

The relative uncertainty is one over the square root of the count; the absolute one is the
concentration times that. A bin with no particles has no Poisson estimate and is `NaN`, not zero.

In [3]:
rel = pu.per_bin_relative_uncertainty(count)
ab = pu.per_bin_absolute_uncertainty(conc, count)
good = count > 0
assert np.allclose(rel[good], 1 / np.sqrt(count[good]))
assert np.allclose(ab[good], conc[good] / np.sqrt(count[good]))
assert np.isnan(rel[~good & ~np.isnan(count)]).all()      # zero-count bins: no estimate
assert np.isnan(rel[17]).all()                            # uncovered second stays NaN
print(f"median relative uncertainty, bin 0: {np.nanmedian(rel[:, 0]):.3f}; "
      f"bin 10: {np.nanmedian(rel[:, 10]):.3f}; "
      f"zero-count bins: {int((count == 0).sum())}")

median relative uncertainty, bin 0: 0.050; bin 10: 0.302; zero-count bins: 107


## Which bins are statistically significant

The field convention treats 100 particles in a bin as statistically significant. The flag is
`False` both for a bin below the threshold and for an uncovered second; the threshold can be changed.

In [4]:
sig = pu.is_statistically_significant(count)
assert sig.dtype == bool
assert (sig == (np.nan_to_num(count, nan=-1) >= pu.STATISTICALLY_SIGNIFICANT_COUNT)).all()
assert not sig[17].any()
sig10 = pu.is_statistically_significant(count, threshold=10)
assert sig10.sum() >= sig.sum()
print("significant bins per second (median), threshold 100:", int(np.median(sig.sum(axis=1))),
      "; threshold 10:", int(np.median(sig10.sum(axis=1))))

significant bins per second (median), threshold 100: 4 ; threshold 10: 11


## Uncertainty of the per-second total

Summing over bins gives the total count per second, whose relative uncertainty is one over the
square root of the total. Bins with `NaN` contribute nothing; a second with no coverage at all stays
`NaN` rather than becoming zero.

In [5]:
total, rel_tot, abs_tot = pu.propagate_to_total(count, axis=1)
ok = ~np.isnan(total)
assert np.isnan(total[17]) and np.isnan(rel_tot[17])
assert np.allclose(total[ok], np.nansum(count[ok], axis=1))
assert np.allclose(abs_tot[ok], np.sqrt(total[ok]))
# the same figure by adding the per-bin absolute count uncertainties in quadrature
quad = np.sqrt(np.nansum(np.where(count > 0, count, 0.0), axis=1))
assert np.allclose(quad[ok], abs_tot[ok])
print(f"median total {np.median(total[ok]):.0f} particles/s, relative uncertainty {np.median(rel_tot[ok]):.4f}")

median total 1356 particles/s, relative uncertainty 0.0272


## Checking the estimate against repeated draws

For a known mean, the spread of many independent one-second draws should match the one-over-root-N
estimate made from a single draw. Two thousand draws per bin are compared bin by bin.

In [6]:
draws = rng.poisson(mean_per_bin, size=(2000, n_bin)).astype(float)
observed_rel_spread = draws.std(axis=0) / draws.mean(axis=0)
predicted = np.nanmedian(pu.per_bin_relative_uncertainty(draws), axis=0)
check = mean_per_bin >= 20                       # bins with enough particles for a stable spread
ratio = observed_rel_spread[check] / predicted[check]
print("observed / predicted relative spread:", np.round(ratio, 3))
assert np.all(np.abs(ratio - 1) < 0.1)

observed / predicted relative spread: [0.974 1.002 1.002 1.004 0.992 0.972 0.985 1.012 0.995]


## Summary

- `per_bin_relative_uncertainty` returns one over the square root of the count, and `NaN` for an
  empty bin or an uncovered second.
- `per_bin_absolute_uncertainty` scales that by the concentration.
- `is_statistically_significant` flags bins with at least 100 particles (adjustable), and never
  flags an uncovered second.
- `propagate_to_total` gives the per-second total and its uncertainty, equal to adding the per-bin
  uncertainties in quadrature, and keeps an uncovered second as `NaN`.
- On repeated draws, the observed scatter matches the estimate within 10 % in every bin with a mean
  of at least 20 particles.
- None of these change a stored value; they add arrays beside the product.